# سبق 04 - ٹول استعمال کرنے کا ڈیزائن پیٹرن

اس سبق میں آپ Microsoft Agent Framework (Python) کا استعمال کرتے ہوئے AI ایجنٹس کے لیے **ٹول استعمال** ڈیزائن پیٹرن سیکھیں گے۔ ہم درج ذیل موضوعات کا احاطہ کریں گے:

- `@tool` ڈیکوریٹر اور ٹائپ کیے ہوئے پیرامیٹرز کے ساتھ فنکشن ٹولز کی تعریف کرنا
- ماڈل کو معلوم ہو کہ ہر ٹول کیا کرتا ہے، اس کے لیے ٹول اسکیمہ فراہم کرنا
- `approval_mode` کے ذریعے ٹول کے عملدرآمد کو کنٹرول کرنا
- Pydantic ماڈلز اور `response_format` کے ذریعے **ساختہ شدہ آؤٹ پٹ** فراہم کرنا

یہ منظر نامہ ایک **ٹریول بکنگ ایجنٹ** کا ہے جو مقامات کی تلاش کر سکتا ہے، دستیابی چیک کر سکتا ہے، اور پرواز کی معلومات حاصل کر سکتا ہے۔


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ڈیکوریٹر کے ساتھ ٹولز کی تعریف کرنا

`@tool` ڈیکوریٹر سادہ پائیتھون فنکشن کو ایک ایسے ٹول میں تبدیل کر دیتا ہے جسے ایجنٹ کال کر سکتا ہے۔
اہم نکات:

- **ڈوک سٹرنگ** وہ ٹول کی وضاحت بن جاتی ہے جو ماڈل دیکھتا ہے۔
- **قسم کی تشریحات** (جس میں وضاحتوں کے ساتھ `Annotated` شامل ہے) ٹول کے نقشہ کو متعین کرتی ہیں۔
- `approval_mode` کنٹرول کرتا ہے کہ آیا صارف کو ہر کال کو عمل سے پہلے منظوری دینی ہوگی۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایک ایجنٹ بنانا جس میں متعدد ٹولز ہوں

تمام تینوں ٹولز کو کلائنٹ کو پاس کریں تاکہ ماڈل صارف کے سوال کا جواب دینے کے لیے جو بھی ٹولز درکار ہوں ان کو بلا سکے۔


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ٹولز کے ساتھ منظم آؤٹ پٹ

`response_format` کو ایک پائیڈینٹک ماڈل پر سیٹ کر کے، ایجنٹ کو مجبور کیا جاتا ہے کہ وہ آزادانہ شکل کے متن کی بجائے ایک اچھی طرح سے ٹائپ کیا ہوا JSON آبجیکٹ واپس کرے۔ یہ اس وقت مفید ہوتا ہے جب نیچے کے کوڈ کو نتیجہ پروگرامنگ طریقے سے استعمال کرنا ہو۔


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## آلات کی منظوری کے انداز

`@tool` پر `approval_mode` پیرامیٹر اس بات کا کنٹرول کرتا ہے کہ آیا آلات کی کالز کو چلانے سے پہلے انسانی منظوری درکار ہے:

| انداز | رویہ |
|---|---|
| `"never_require"` | آلہ خودکار طور پر چلتا ہے — صارف کی تصدیق کی ضرورت نہیں۔ |
| `"always_require"` | ہر کال کو چلانے سے پہلے صارف کی منظوری ضروری ہے۔ |

ان آلات کے لیے `"always_require"` استعمال کریں جن کے ضمنی اثرات ہوتے ہیں (جیسے پرواز کی بکنگ، کریڈٹ کارڈ چارج کرنا) تاکہ انسانی مداخلت برقرار رہے۔


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **آلات کی تعریف کریں** `@tool` ڈیکوریٹر کا استعمال کرتے ہوئے جس میں ٹائپ کیے گئے پیرامیٹرز اور ڈاک اسٹرنگز شامل ہوتی ہیں جو آلے کے اسکیما کے طور پر کام کرتی ہیں۔
2. **متعدد آلات کو ترتیب دیں** تاکہ ایجنٹ انہیں پیچیدہ سوالات کے جوابات دینے کے لیے تسلسل سے کال کر سکے۔
3. **مربوط نتیجہ واپس کریں** پائڈینٹک ماڈل کو `response_format` کے طور پر پاس کر کے۔
4. **آلے کی منظوری کو کنٹرول کریں** `approval_mode` کے ذریعے تاکہ حساس عمل کے لیے انسان کو عمل دخل میں رکھا جا سکے۔

یہ پیٹرنز قابل اعتماد، پیداوار کے قابل ایجنٹس بنانے کی بنیاد فراہم کرتے ہیں جو باہر کے نظاموں کے ساتھ محفوظ طریقے سے تعامل کر سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
